In [1]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd
import mlflow
import mlflow.sklearn
import optuna
import math
import pathlib
import pickle
from optuna.samplers import TPESampler
from sklearn.metrics import f1_score, precision_score, recall_score
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("/home/josue/Documents/Data_Science_Final_Project/data/raw/adult.csv")
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [3]:
def preprocessor(df):
    # Target binario
    y = df["income"].apply(lambda x: 1 if ">50K" in x else 0)

    # Features (quitamos income y education)
    X = df.drop(["income", "education"], axis=1)

    # Columnas categóricas y numéricas
    categorical_cols = [
        'workclass', 'marital.status', 
        'occupation', 'race', 'relationship', 
        'sex', 'native.country'
    ]
    numeric_cols = [col for col in X.columns if col not in categorical_cols]

    # Transformadores
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    # Pipeline de preprocesamiento
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

    # Split inicial (train + temp)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    # Split secundario (val + test)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    # Ajustar y transformar solo con train
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)
    X_test_processed = preprocessor.transform(X_test)

    # Obtener nombres de columnas
    cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
    all_features = numeric_cols + cat_features.tolist()

    # Reconstruir DataFrames
    X_train_processed = pd.DataFrame(X_train_processed, columns=all_features, index=X_train.index)
    X_val_processed = pd.DataFrame(X_val_processed, columns=all_features, index=X_val.index)
    X_test_processed = pd.DataFrame(X_test_processed, columns=all_features, index=X_test.index)

    return X_train_processed, X_val_processed, X_test_processed, y_train, y_val, y_test

X_train_processed, X_val_processed, X_test_processed, y_train, y_val, y_test = preprocessor(df)


## Random forest classifier

In [4]:


# ------------------------------------------------------------
# Definición del objetivo para Optuna
# ------------------------------------------------------------
def objective_rf(trial, X_train, y_train, X_val, y_val):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": 42,
        "n_jobs": -1
    }

    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "RandomForestClassifier")
        mlflow.log_params(params)

        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)

        mlflow.log_metrics({
            "f1": f1,
            "precision": precision,
            "recall": recall
        })

        
        input_example = X_val.head(5)
        signature = infer_signature(input_example, y_val.head(5))

        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            input_example=input_example,
            signature=signature
        )

    # Queremos maximizar F1
    return 1 - f1



In [ ]:

mlflow.sklearn.autolog(log_models=False)

sampler = TPESampler(seed=42)
study_rf = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="RandomForest Optuna Optimization"):
    study_rf.optimize(lambda trial: objective_rf(trial, X_train_processed, y_train, X_val_processed, y_val),
                      n_trials=10)

    best_params = study_rf.best_params
    best_params["random_state"] = 42
    mlflow.log_params(best_params)

    mlflow.set_tags({
        "project": "Adult Income Prediction",
        "optimizer_engine": "Optuna",
        "model_family": "RandomForestClassifier"
    })

    final_model = RandomForestClassifier(**best_params)
    final_model.fit(X_train_processed, y_train)

    y_pred = final_model.predict(X_test_processed)
    f1 = f1_score(y_test, y_pred)
    mlflow.log_metric("final_f1", f1)

    # Guardar preprocesador
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(preprocessor, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    # Registrar modelo final
    input_example = X_test_processed.head(5)
    signature = infer_signature(input_example, y_test.head(5))

    mlflow.sklearn.log_model(
        sk_model=final_model,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )


[I 2025-11-11 17:08:23,229] A new study created in memory with name: no-name-fc97a461-96f6-45a5-98e0-8cd65275c23f
/home/josue/anaconda3/envs/ITESO/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 17:08:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[I 20

## Gradient Boost Classifier

In [ ]:


def objective_gb(trial, X_train, y_train, X_val, y_val):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "random_state": 42
    }

    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "GradientBoostingClassifier")
        mlflow.log_params(params)

        model = GradientBoostingClassifier(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)

        mlflow.log_metrics({"f1": f1, "precision": precision, "recall": recall})

        input_example = X_val.head(5)
        signature = infer_signature(input_example, y_val.head(5))
        mlflow.sklearn.log_model(model, "model", input_example=input_example, signature=signature)

    return 1 - f1
    


In [ ]:
mlflow.sklearn.autolog(log_models=False)
sampler = TPESampler(seed=42)
study_gb = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="GradientBoosting Optuna Optimization"):
    study_gb.optimize(lambda trial: objective_gb(trial, X_train_processed, y_train, X_val_processed, y_val),
                      n_trials=10)

    best_params = study_gb.best_params
    best_params["random_state"] = 42
    mlflow.log_params(best_params)

    mlflow.set_tags({
        "project": "Adult Income Prediction",
        "optimizer_engine": "Optuna",
        "model_family": "GradientBoostingClassifier"
    })

    final_model = GradientBoostingClassifier(**best_params)
    final_model.fit(X_train_processed, y_train)

    y_pred = final_model.predict(X_test_processed)
    f1 = f1_score(y_test, y_pred)
    mlflow.log_metric("final_f1", f1)

    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(preprocessor, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    input_example = X_test_processed.head(5)
    signature = infer_signature(input_example, y_test.head(5))
    mlflow.sklearn.log_model(final_model, "model", input_example=input_example, signature=signature)


## XBG Classifier

In [ ]:


def objective_xgb(trial, X_train, y_train, X_val, y_val):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "use_label_encoder": False,
        "eval_metric": "logloss"
    }

    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "XGBClassifier")
        mlflow.log_params(params)

        model = XGBClassifier(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)

        mlflow.log_metrics({"f1": f1, "precision": precision, "recall": recall})

        input_example = X_val.head(5)
        signature = infer_signature(input_example, y_val.head(5))
        mlflow.sklearn.log_model(model, "model", input_example=input_example, signature=signature)

    return 1 - f1



In [ ]:

mlflow.sklearn.autolog(log_models=False)
sampler = TPESampler(seed=42)
study_xgb = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="XGBoost Optuna Optimization"):
    study_xgb.optimize(lambda trial: objective_xgb(trial, X_train_processed, y_train, X_val_processed, y_val),
                       n_trials=10)

    best_params = study_xgb.best_params
    best_params["random_state"] = 42
    mlflow.log_params(best_params)

    mlflow.set_tags({
        "project": "Adult Income Prediction",
        "optimizer_engine": "Optuna",
        "model_family": "XGBClassifier"
    })

    final_model = XGBClassifier(**best_params)
    final_model.fit(X_train_processed, y_train)

    y_pred = final_model.predict(X_test_processed)
    f1 = f1_score(y_test, y_pred)
    mlflow.log_metric("final_f1", f1)

    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(preprocessor, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    input_example = X_test_processed.head(5)
    signature = infer_signature(input_example, y_test.head(5))
    mlflow.sklearn.log_model(final_model, "model", input_example=input_example, signature=signature)
